In [33]:
!rm -rf NN-Project1/

In [34]:
!git clone https://www.github.com/yousefkoriem/NN-Project1.git

Cloning into 'NN-Project1'...
remote: Enumerating objects: 295, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 295 (delta 6), reused 46 (delta 4), pack-reused 247 (from 2)
Receiving objects: 100% (295/295), 241.55 MiB | 31.25 MiB/s, done.
Resolving deltas: 100% (66/66), done.


In [35]:
import os
os.chdir("/content/NN-Project1")

In [36]:
import keras
import tensorflow as tf
import numpy as np
import pandas as pd
from keras import layers
from keras.utils import text_dataset_from_directory
import spacy
import re
import pickle
from keras import layers, models, metrics, optimizers,callbacks

In [37]:
model = models.load_model("models/architecture/untrained_imdb_model.keras")

In [38]:
checkpoint = callbacks.ModelCheckpoint(
    filepath="models/architecture/best_imdb_model.keras",
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

In [39]:
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

In [40]:
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [41]:
# 1. Load the raw datasets
train_ds = tf.data.Dataset.load("data/processed/train_ds")
val_ds = tf.data.Dataset.load("data/processed/val_ds")
test_ds = tf.data.Dataset.load("data/processed/test_ds")

# 2. Define the label shape fix
def vectorize_text(text, label):
    return text, tf.expand_dims(label, -1)

# 3. Map the fix to the data
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(vectorize_text, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.map(vectorize_text, num_parallel_calls=AUTOTUNE)
test_ds = test_ds.map(vectorize_text, num_parallel_calls=AUTOTUNE)

# 4. Apply cache and prefetch exactly ONCE at the very end
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [42]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

Epoch 1/100
620/625 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6510 - f1_score: 0.6835 - loss: 0.5793
Epoch 1: val_loss improved from None to 0.31242, saving model to models/architecture/best_imdb_model.keras

Epoch 1: finished saving model to models/architecture/best_imdb_model.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.7653 - f1_score: 0.7721 - loss: 0.4545 - val_accuracy: 0.8682 - val_f1_score: 0.8700 - val_loss: 0.3124 - learning_rate: 0.0010
Epoch 2/100
621/625 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8897 - f1_score: 0.8906 - loss: 0.2725
Epoch 2: val_loss did not improve from 0.31242
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.9168 - f1_score: 0.9166 - loss: 0.2177 - val_accuracy: 0.8736 - val_f1_score: 0.8757 - val_loss: 0.3194 - learning_rate: 0.0010
Epoch 3/100
621/625 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9519 - f1_score: 0.9522 - loss: 0.1346
Epoch 3: val_loss did not improve from 0.31242
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/

In [ ]:
model.summary()

In [ ]:
!mkdir results/tables

In [ ]:
history_df = pd.DataFrame(history.history)
history_df.to_csv("results/tables/history.csv", index=False)

In [ ]:
score = model.evaluate(test_ds)
score_df = pd.DataFrame([score], columns=["loss", "accuracy", "f1_score"])

782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8568 - f1_score: 0.8491 - loss: 0.3266


In [ ]:
!mkdir results/models

In [ ]:
model.save("results/models/final_imdb_model.keras")